<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 11 · Event-Based Backtesting Framework

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook explores the event-based backtester introduced in Chapter 11.

- Inspect the event types and their relationships.
- Run a short SMA-based backtest using the event loop.
- Compare event-driven and vectorised equity curves.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})

PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch11_event_backtester as ch11  # event-based framework
import ch05_eod_engineering as ch05  # price data helper
import ch07_baseline_strategies as ch07  # vectorised benchmark


### 1. Event Types at a Glance

The snippet below lists the dataclass-style events used in the engine. They show how market data, signals, orders, and fills are separated conceptually.

In [ ]:
for cls in (ch11.MarketEvent, ch11.SignalEvent, ch11.OrderEvent, ch11.FillEvent):
    print(cls.__name__, cls.__annotations__)


### 2. Running a Short Event-Based Backtest

We reuse the helpers from the script to run an SMA crossover strategy on a truncated price history, which keeps the event loop compact for inspection.

In [ ]:
panel = ch05.load_eod_panel()
prices_eur = panel["EURUSD"].astype(float).dropna().iloc[-750:]

data_handler = ch11.HistoricalDataHandler(symbol="EURUSD")
data_handler.prices = prices_eur
data_handler.index = prices_eur.index
data_handler.pointer = 0

events = ch11.EventQueue()
strategy = ch11.SmaCrossStrategy(fast_window=20, slow_window=100)
portfolio = ch11.NaivePortfolio()
execution = ch11.SimulatedExecutionHandler()

while data_handler.have_more_bars():
    data_handler.stream_next(events)
    while not events.empty():
        event = events.get()
        if isinstance(event, ch11.MarketEvent):
            execution.on_market_event(event)
            strategy.on_market_event(event, events)
        elif isinstance(event, ch11.SignalEvent):
            portfolio.on_signal(event, events)
        elif isinstance(event, ch11.OrderEvent):
            execution.on_order(event, events)
        elif isinstance(event, ch11.FillEvent):
            portfolio.on_fill(event)

pos_event = portfolio.get_position_series()
pos_event.head()


### 3. Vectorised vs. Event-Driven Equity Curves

Using the same cost model, we compare the event-driven results with the vectorised SMA helper from Chapter 7.

In [ ]:
pos_vec = ch07.build_positions_sma(prices_eur)

net_vec = ch07.backtest_strategy(prices_eur, pos_vec)
net_event = ch07.backtest_strategy(prices_eur, pos_event)

wealth_vec = (1.0 + net_vec).cumprod()
wealth_event = (1.0 + net_event).cumprod()

fig, ax = plt.subplots(figsize=(7.0, 3.8))
ax.plot(wealth_vec.index, wealth_vec.values, label="vectorised SMA")
ax.plot(wealth_event.index, wealth_event.values, label="event-based SMA")
ax.set_ylabel("wealth (normalised)")
ax.set_xlabel("date")
ax.set_title("EURUSD: vectorised vs. event-based SMA backtest")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Takeaways

- Event-based backtesting makes the flow of information and orders explicit while still matching vectorised results.
- Separating events, strategies, portfolios, and execution handlers clarifies where to plug in streaming data and broker APIs later on.
- The same cost model and diagnostics can be shared between vectorised and event-driven engines.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">